# Baseline Model Testing: YouTube Engagement Score Prediction (Regression)

This notebook implements baseline model testing for predicting continuous engagement scores on Algerian Baccalaureate educational videos.

## Objectives
1. Load engineered features
2. Create train/validation/test splits
3. Train multiple baseline regression models
4. Evaluate performance using regression metrics (MAE, RMSE, R²)
5. Analyze feature importance
6. **Save data splits to data/modeling/**

## Step 1: Setup and Imports

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
from datetime import datetime
import warnings
import sys
import os
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
# Also set Python's built-in random if used
try:
    import random
    random.seed(RANDOM_SEED)
except ImportError:
    pass

# Fix path resolution - get project root
NOTEBOOK_DIR = Path.cwd() if Path.cwd().name == 'notebooks' else Path(__file__).parent if '__file__' in globals() else Path.cwd()
if NOTEBOOK_DIR.name != 'notebooks':
    # Try to find notebooks directory
    for parent in NOTEBOOK_DIR.parents:
        if (parent / 'notebooks').exists():
            NOTEBOOK_DIR = parent / 'notebooks'
            break
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / 'data'

# Add project root to path for imports
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"📁 Project root: {PROJECT_ROOT}")
print(f"📁 Data directory: {DATA_DIR}")
print(f"🎲 Random seed set to {RANDOM_SEED} for reproducibility")

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
# Note: Regression metrics (MAE, RMSE, R²) will be imported when needed in Step 7

# Ensemble model availability flags (optional - install xgboost and lightgbm)
try:
    from xgboost import XGBRegressor
    XGBOOST_AVAILABLE = True
except ImportError:
    print("⚠️ XGBoost not available. Install with: uv add xgboost")
    XGBOOST_AVAILABLE = False

try:
    from lightgbm import LGBMRegressor
    LIGHTGBM_AVAILABLE = True
except ImportError:
    print("⚠️ LightGBM not available. Install with: uv add lightgbm")
    LIGHTGBM_AVAILABLE = False

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Imports complete")

## Step 2: Load Engineered Data

In [ ]:
# Load engineered features - try multiple paths
engineered_path = DATA_DIR / "processed" / "videos_engineered.csv"
bac_only_path = DATA_DIR / "processed" / "videos_bac_only.csv"

if engineered_path.exists():
    print(f"✅ Loading from: {engineered_path}")
    df = pd.read_csv(engineered_path)
elif bac_only_path.exists():
    print(f"⚠️ videos_engineered.csv not found. Loading {bac_only_path} and running feature engineering...")
    from src.features.engineer import VideoFeatureEngineer
    
    df_bac = pd.read_csv(bac_only_path)
    print(f"   Loaded {len(df_bac):,} Bac videos")
    
    # Run feature engineering
    engineer = VideoFeatureEngineer()
    df = engineer.fit_transform(df_bac)
    df = engineer.select_features(df, include_target=True)
    
    print(f"✅ Feature engineering complete: {df.shape[1]} features")
    
    # Save for future use
    engineered_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(engineered_path, index=False)
    print(f"💾 Saved to: {engineered_path}")
else:
    raise FileNotFoundError(
        f"❌ Neither {engineered_path} nor {bac_only_path} found.\n"
        f"   Run: uv run python run_pipeline.py filter_data --validate\n"
        f"   Then: uv run python -m src.features.build_features --input {DATA_DIR / 'processed' / 'videos_bac_only.csv'}"
    )

# Verify required columns
required_cols = ['engagement_score', 'channel_id']
missing_cols = [col for col in required_cols if col not in df.columns]
if missing_cols:
    raise ValueError(f"❌ Missing required columns: {missing_cols}")

print(f"\n📊 Loaded {len(df):,} videos")
print(f"Features: {df.shape[1]} columns")
print(f"\nEngagement score statistics:")
print(df['engagement_score'].describe())
if 'subject' in df.columns:
    print(f"\nSubject distribution:")
    print(df['subject'].value_counts())

# Display first few rows
df.head()

# Step 2.5: Data Cleaning (Handle Missing Values)


In [ ]:
# 1. Add transcript availability indicator
transcript_cols = ['transcript_word_count', 'transcript_char_count', 'transcript_sentence_count']
df['has_transcript'] = df[transcript_cols[0]].notna().astype(int)
print(f"📊 Videos with transcripts: {df['has_transcript'].sum():,} ({df['has_transcript'].mean()*100:.1f}%)")

# 2. Fill ALL transcript-derived features with 0 (comprehensive list)
transcript_feature_patterns = [
    'transcript', 'flesch', 'fog', 'readability', 'speech_rate', 'lexical', 
    'sentence', 'question', 'example', 'explanation', 'contrast', 
    'keyword_count', 'keyword_density', 'technical_term', 'unique_word',
    'domain_keyword', 'bac_marker', 'pedagogical_marker',  # NEW features
    'speech_rate_above', 'speech_rate_below', 'speech_rate_optimal'  # Pacing flags
]

transcript_feature_cols = [col for col in df.columns if any(x in col.lower() for x in transcript_feature_patterns)]
for col in transcript_feature_cols:
    if col in df.columns and df[col].dtype in ['float64', 'int64', 'float32', 'int32']:
        df[col] = df[col].fillna(0)

# 3. Fill description NaN
df['description'] = df['description'].fillna('')
df['description_length'] = df['description'].str.len()

# 4. Fill any remaining numeric columns with 0 (or median for specific cases)
numeric_cols = df.select_dtypes(include=['float64', 'int64', 'float32', 'int32']).columns
for col in numeric_cols:
    if df[col].isnull().sum() > 0:
        df[col] = df[col].fillna(0)
        print(f"⚠️  Filled {col} NaN with 0")

# 5. Report cleaning results
remaining_nan = df.isnull().sum().sum()
print(f"✅ Filled NaN in {len(transcript_feature_cols)} transcript columns with 0")
print(f"✅ Remaining NaN values: {remaining_nan}")
if remaining_nan > 0:
    nan_cols = df.columns[df.isnull().any()].tolist()
    print(f"⚠️  NaN remaining in: {nan_cols}")

# Step 2.6: Final Imputation and Data Export 
Based on Data_cleaning_guide.md recommendations

In [ ]:


print("🔧 Final Imputation Pass")
print("=" * 60)

# 1. Text fields -> empty string
df['description'] = df['description'].fillna('')
df['tags'] = df['tags'].fillna('') if 'tags' in df.columns else ''

# 2. Transcript-derived numerical features -> 0
# (For videos without transcripts, these should be 0, not NaN)
transcript_numeric_cols = [
    'transcript_word_count', 'transcript_char_count', 'transcript_sentence_count',
    'unique_word_count', 'lexical_diversity', 'avg_words_per_sentence',
    'flesch_reading_ease', 'flesch_kincaid_grade', 'gunning_fog_index',
    'automated_readability_index', 'speech_rate_wpm', 'speech_rate_optimal',
    'question_count', 'question_density', 'example_count', 'example_density',
    'explanation_count', 'explanation_density', 'contrast_count', 'contrast_density',
    'technical_term_count', 'technical_term_density', 
    'subject_keyword_count', 'subject_keyword_density'
]

for col in transcript_numeric_cols:
    if col in df.columns:
        missing = df[col].isna().sum()
        if missing > 0:
            df[col] = df[col].fillna(0)
            print(f"  ✓ {col}: {missing:,} NaN → 0")

# 3. Verify no remaining NaN in key columns
remaining_nan = df.isnull().sum()
cols_with_nan = remaining_nan[remaining_nan > 0]
if len(cols_with_nan) > 0:
    print(f"\n⚠️ Remaining NaN columns:")
    print(cols_with_nan.to_string())
else:
    print(f"\n✅ No remaining NaN values!")

print(f"\n📊 Final dataset shape: {df.shape}")

# Step 2.7: Save Final Imputed Dataset


In [ ]:
# ============================================================
# Step 2.7: Save Final Imputed Dataset
# ============================================================

output_path = DATA_DIR / "cleaned" / "data_final.csv"
output_path.parent.mkdir(parents=True, exist_ok=True)

# Save the fully imputed dataset
df.to_csv(output_path, index=False)

print(f"💾 Saved final imputed dataset to: {output_path}")
print(f"   Shape: {df.shape}")
print(f"   Size: {output_path.stat().st_size / (1024*1024):.2f} MB")

# Verification
df_check = pd.read_csv(output_path, nrows=5)
print(f"   ✓ File readable, columns: {len(df_check.columns)}")

## Step 3: Feature Preparation

In [ ]:
# Define target and features
target_col = 'engagement_score'

# Exclude columns to prevent data leakage and remove non-features
exclude_cols = [
    'video_id', 'channel_id', 'channel_title',
    target_col, 'engagement_category',
    'view_count', 'like_count', 'comment_count',
    'like_ratio', 'comment_ratio',
    'channel_avg_engagement',
    'title', 'description', 'tags', 'publish_date', 'publish_day_of_week',
    "channel_video_avg_duration", 
    #"channel_video_count", 
    #"channel_avg_views", 
    #"channel_age_days"
]

# Get feature columns
feature_cols = [col for col in df.columns if col not in exclude_cols]

# Identify categorical and numerical features
categorical_features = [col for col in feature_cols if df[col].dtype == 'object']
numerical_features = [col for col in feature_cols if col not in categorical_features]

# Verify numerical features
for col in numerical_features:
    if not pd.api.types.is_numeric_dtype(df[col]):
        print(f"⚠️ Warning: {col} is not numeric but in numerical_features")

print(f"🎯 Target: {target_col}")
print(f"📊 Total features: {len(feature_cols)}")
print(f"   - Numerical: {len(numerical_features)}")
print(f"   - Categorical: {len(categorical_features)}")

print(f"\n✅ Included features:")

temporal = [col for col in numerical_features if any(x in col for x in ['days_since', 'publish_hour', 'publish_month', 'publish_year', 'is_evening', 'is_weekday'])]
print(f"\nTemporal features ({len(temporal)}):")
print(temporal)

text_extracted = [col for col in numerical_features if any(x in col for x in ['length', 'word_count', 'tag_count', 'is_exam'])]
print(f"\nText extraction features ({len(text_extracted)}):")
print(text_extracted)

channel_features = [col for col in numerical_features if 'channel' in col]
print(f"\nChannel features ({len(channel_features)}):")
print(channel_features)

content_features = [col for col in numerical_features if col in ['duration_sec']]
print(f"\nContent features ({len(content_features)}):")
print(content_features)

# Catch-all for any other numerical features (e.g., TF-IDF, PCA components)
categorized = set(temporal + text_extracted + channel_features + content_features)
other_numerical = [col for col in numerical_features if col not in categorized]
if other_numerical:
    print(f"\nOther numerical features ({len(other_numerical)}):")
    print(other_numerical)

print(f"\nCategorical features ({len(categorical_features)}):")
print(categorical_features)

print(f"\n❌ Excluded (to prevent data leakage):")
excluded_present = [col for col in exclude_cols if col in df.columns]
print(excluded_present)

In [ ]:
# One-hot encode categorical features
# IMPORTANT: Use sorted categories to ensure consistent column ordering for reproducibility
df_encoded = pd.get_dummies(df, columns=categorical_features, drop_first=True)

# Reorder all columns alphabetically for reproducibility (ensures consistent feature matrix)
# This prevents feature shuffling when data is loaded in different order
df_encoded = df_encoded.reindex(sorted(df_encoded.columns), axis=1)

# Update feature columns after encoding - use SAME exclude_cols to ensure consistency
# This ensures text columns, raw counts, and derived engagement metrics are excluded
encoded_feature_cols = [col for col in df_encoded.columns if col not in exclude_cols]
# Sort feature columns alphabetically for reproducibility
encoded_feature_cols = sorted(encoded_feature_cols)

# Verify all encoded features are numeric (after one-hot encoding)
non_numeric = [col for col in encoded_feature_cols if not pd.api.types.is_numeric_dtype(df_encoded[col])]
if non_numeric:
    print(f"⚠️ Warning: Non-numeric columns found: {non_numeric}")
    print("   These will be excluded from feature set.")
    encoded_feature_cols = [col for col in encoded_feature_cols if pd.api.types.is_numeric_dtype(df_encoded[col])]
    encoded_feature_cols = sorted(encoded_feature_cols)  # Re-sort after filtering

print(f"✅ After encoding: {len(encoded_feature_cols)} features")
print(f"\nNew subject dummy columns:")
subject_dummies = [col for col in encoded_feature_cols if col.startswith('subject_')]
print(subject_dummies)

print(f"\n✅ All features are numeric (ready for scaling)")
print(f"   Sample features: {encoded_feature_cols[:5]}...")

## Step 3.5: VIF-Based Feature Selection (Reduce Multicollinearity)

In [ ]:
# ============================================================
# VIF-Based Feature Selection
# Removes highly correlated features to reduce multicollinearity
# ============================================================

from statsmodels.stats.outliers_influence import variance_inflation_factor

def select_features_vif(X, vif_threshold=10.0, correlation_threshold=0.95):
    """
    Select features using VIF and correlation analysis.
    
    Args:
        X: DataFrame of features
        vif_threshold: Max VIF (remove if higher)
        correlation_threshold: Max inter-feature correlation
        
    Returns:
        List of selected feature names
    """
    X = X.copy().fillna(0)
    
    # Step 1: Remove highly correlated features
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    
    to_drop = set()
    for col in upper.columns:
        highly_correlated = upper.index[upper[col] > correlation_threshold].tolist()
        to_drop.update(highly_correlated)
    
    if to_drop:
        print(f"📉 Removing {len(to_drop)} features due to high correlation (>{correlation_threshold}):")
        print(f"   {list(to_drop)[:10]}{'...' if len(to_drop) > 10 else ''}")
        X = X.drop(columns=list(to_drop))
    
    # Step 2: Iterative VIF removal
    removed = []
    while len(X.columns) >= 2:
        vif_data = []
        for i, col in enumerate(X.columns):
            try:
                vif = variance_inflation_factor(X.values, i)
                vif_data.append((col, vif))
            except:
                vif_data.append((col, 0))
        
        vif_df = pd.DataFrame(vif_data, columns=['feature', 'VIF'])
        max_vif = vif_df.loc[vif_df['VIF'].idxmax()]
        
        if max_vif['VIF'] > vif_threshold:
            removed.append(max_vif['feature'])
            X = X.drop(columns=[max_vif['feature']])
        else:
            break
    
    if removed:
        print(f"📉 Removed {len(removed)} features due to high VIF (>{vif_threshold}):")
        print(f"   {removed[:10]}{'...' if len(removed) > 10 else ''}")
    
    print(f"\n✅ Final feature count: {len(X.columns)}")
    return X.columns.tolist()

# Prepare feature matrix
X_all = df_encoded[encoded_feature_cols].copy()

# Run VIF selection
print("🔍 Running VIF-based feature selection...")
print(f"   Starting with {len(encoded_feature_cols)} features")
selected_features = select_features_vif(X_all, vif_threshold=10.0, correlation_threshold=0.95)

# Update feature set
print(f"\n📊 Features retained: {len(selected_features)}/{len(encoded_feature_cols)}")

In [ ]:
# Show which features were kept vs dropped
kept_set = set(selected_features)
dropped_features = [f for f in encoded_feature_cols if f not in kept_set]

print("✅ KEPT FEATURES:")
for f in sorted(selected_features):
    print(f"   • {f}")

print(f"\n❌ DROPPED FEATURES ({len(dropped_features)}):")
for f in sorted(dropped_features):
    print(f"   • {f}")

## Step 4: Create Train/Val/Test Splits (Within-Channel Splitting)

We split videos randomly across all channels (60% train, 20% val, 20% test).
Since we include channel-level features explicitly (channel_video_count, channel_avg_views, etc.), 
splitting within channels is acceptable for regression.

In [ ]:
# Split videos WITHIN each channel (not by channel)
# This ensures balanced splits: 60% train, 20% val, 20% test
# Since we include channel-level features explicitly, this is acceptable

print(f"📺 Total unique channels: {df_encoded['channel_id'].nunique()}")
print(f"📹 Total videos: {len(df_encoded):,}")

# Strategy: Split videos randomly across all channels
# No stratification needed for regression

# First split: Train+Val (80%) vs Test (20%)
train_val_idx, test_idx = train_test_split(
    df_encoded.index,
    test_size=0.20,
    random_state=RANDOM_SEED
)

# Second split: Train (75% of train_val) vs Val (25% of train_val)
# Final: Train 60%, Val 20%, Test 20%
train_idx, val_idx = train_test_split(
    train_val_idx,
    test_size=0.25,  # 0.25 * 0.80 = 0.20 of total
    random_state=RANDOM_SEED
)

# Create split datasets
train_df = df_encoded.loc[train_idx].copy()
val_df = df_encoded.loc[val_idx].copy()
test_df = df_encoded.loc[test_idx].copy()

print(f"\n✅ Split complete (videos split within channels):")
print(f"Train: {len(train_df):,} videos ({len(train_df)/len(df_encoded)*100:.1f}%)")
print(f"Val: {len(val_df):,} videos ({len(val_df)/len(df_encoded)*100:.1f}%)")
print(f"Test: {len(test_df):,} videos ({len(test_df)/len(df_encoded)*100:.1f}%)")

# Show channel distribution across splits
print(f"\n📺 Channels represented in each split:")
print(f"Train: {train_df['channel_id'].nunique()} unique channels")
print(f"Val: {val_df['channel_id'].nunique()} unique channels")
print(f"Test: {test_df['channel_id'].nunique()} unique channels")

# Note about overlap (expected - same channels appear in all splits)
print(f"\n⚠️ Note: Videos from the same channel appear in different splits.")
print(f"   This is intentional - we're testing within-channel prediction.")

In [ ]:
# Splits already created above - verify and show statistics

# Check engagement score distribution across splits
print("\n📊 Engagement score distribution across splits:")
for split_name, split_df in [("Train", train_df), ("Val", val_df), ("Test", test_df)]:
    print(f"\n{split_name}:")
    print(f"  Mean: {split_df['engagement_score'].mean():.4f}")
    print(f"  Std: {split_df['engagement_score'].std():.4f}")
    print(f"  Min: {split_df['engagement_score'].min():.4f}")
    print(f"  Max: {split_df['engagement_score'].max():.4f}")

# Verify all channels are represented (some overlap is expected)
all_channels_train = set(train_df['channel_id'].unique())
all_channels_val = set(val_df['channel_id'].unique())
all_channels_test = set(test_df['channel_id'].unique())

channels_in_all_splits = all_channels_train & all_channels_val & all_channels_test
unique_channels = sorted(df_encoded['channel_id'].unique())  # For compatibility with later cells
print(f"\n✅ Channels appearing in all splits: {len(channels_in_all_splits)}/{len(unique_channels)}")
print(f"   This ensures we're testing within-channel generalization.")

## Step 5: Save Data Splits to data/modeling/

**Critical**: Save train/val/test splits for reproducibility and future use

In [ ]:
# Create modeling directory
modeling_dir = DATA_DIR / "modeling"
modeling_dir.mkdir(parents=True, exist_ok=True)

# Save splits as CSV
train_df.to_csv(modeling_dir / "train.csv", index=False)
val_df.to_csv(modeling_dir / "val.csv", index=False)
test_df.to_csv(modeling_dir / "test.csv", index=False)

print("💾 Saved data splits:")
print(f"  - {modeling_dir / 'train.csv'}")
print(f"  - {modeling_dir / 'val.csv'}")
print(f"  - {modeling_dir / 'test.csv'}")

# Save split mapping (video_id to split assignment)
split_mapping = pd.DataFrame({
    'video_id': list(train_df['video_id']) + list(val_df['video_id']) + list(test_df['video_id']),
    'split': ['train'] * len(train_df) + ['val'] * len(val_df) + ['test'] * len(test_df)
})
split_mapping.to_csv(modeling_dir / "splits_mapping.csv", index=False)
print(f"  - {modeling_dir / 'splits_mapping.csv'}")

# Save metadata
metadata = {
    'created_at': datetime.now().isoformat(),
    'total_videos': len(df_encoded),
    'total_channels': len(unique_channels),
    'train_videos': len(train_df),
    'val_videos': len(val_df),
    'test_videos': len(test_df),
    'train_channels': train_df['channel_id'].nunique(),
    'val_channels': val_df['channel_id'].nunique(),
    'test_channels': test_df['channel_id'].nunique(),
    'features': encoded_feature_cols,
    'target': target_col,
    'random_state': 42
}

with open(modeling_dir / "split_metadata.json", 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"  - {modeling_dir / 'split_metadata.json'}")

print("\n✅ All splits saved successfully!")

## Step 6: Prepare X and y for Modeling

In [ ]:
# Prepare features and target using VIF-selected features
# IMPORTANT: Use selected_features (after VIF filtering) instead of encoded_feature_cols

print(f"📊 Using {len(selected_features)} VIF-selected features (reduced from {len(encoded_feature_cols)})")

X_train = train_df[selected_features].values
y_train = train_df[target_col].values

X_val = val_df[selected_features].values
y_val = val_df[target_col].values

X_test = test_df[selected_features].values
y_test = test_df[target_col].values

print(f"\n✅ Prepared datasets (with VIF-selected features):")
print(f"X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"X_val: {X_val.shape}, y_val: {y_val.shape}")
print(f"X_test: {X_test.shape}, y_test: {y_test.shape}")
print(f"\nTarget statistics:")
print(f"Train: Mean={y_train.mean():.4f}, Std={y_train.std():.4f}")
print(f"Val: Mean={y_val.mean():.4f}, Std={y_val.std():.4f}")
print(f"Test: Mean={y_test.mean():.4f}, Std={y_test.std():.4f}")

In [ ]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

print("✅ Features scaled using StandardScaler")

In [ ]:
# ============================================================
# ARABERT SEMANTIC FEATURE EXTRACTION - OPTIMIZED VERSION
# Uses caching to avoid recomputing embeddings
# ============================================================

from src.features.text_embeddings import TextEmbeddingExtractor
import numpy as np
from pathlib import Path
import pickle
import torch
from sklearn.decomposition import PCA

# Check if embeddings are already cached
cache_dir = DATA_DIR / 'modeling'
cache_path = cache_dir / 'arabert_embeddings.pkl'

if cache_path.exists():
    print("📦 Loading cached AraBERT embeddings...")
    with open(cache_path, 'rb') as f:
        cached = pickle.load(f)
        X_train_bert = cached['X_train_bert']
        X_val_bert = cached['X_val_bert']
        X_test_bert = cached['X_test_bert']
    print(f"✅ Loaded cached embeddings: {X_train_bert.shape[1]} dimensions")
else:
    print("📝 Extracting semantic embeddings using AraBERT v2...")
    print("   (pre-trained on Modern Standard Arabic - better for formal educational titles)")
    
    # Check if GPU is available
    device = "cuda" if torch.cuda.is_available() else "cpu"
    if device == "cpu":
        print("   ⚠️ Running on CPU - this will take ~30-45 minutes")
        print("   💡 If you have a GPU, ensure CUDA is installed for 10x speedup\n")
    else:
        print(f"   🚀 Running on GPU ({torch.cuda.get_device_name(0)}) - should take ~2-5 minutes\n")
    
    # IMPORTANT: train_df, val_df, test_df were created from df_encoded (encoded features only)
    # We need to get 'title' and 'description' from the original df DataFrame
    # Create text DataFrames from original df using the same indices
    train_df_text = df.loc[train_idx, ['title', 'description']].copy()
    val_df_text = df.loc[val_idx, ['title', 'description']].copy()
    test_df_text = df.loc[test_idx, ['title', 'description']].copy()
    
    # Combine title + description for richer context
    train_df_text['text_combined'] = (
        train_df_text['title'].fillna('') + ' ' + 
        train_df_text['description'].fillna('')
    )
    val_df_text['text_combined'] = (
        val_df_text['title'].fillna('') + ' ' + 
        val_df_text['description'].fillna('')
    )
    test_df_text['text_combined'] = (
        test_df_text['title'].fillna('') + ' ' + 
        test_df_text['description'].fillna('')
    )
    
    # Load AraBERT model (will download on first run: ~540MB)
    # Defaults to "aubmindlab/bert-base-arabertv2"
    embedding_extractor = TextEmbeddingExtractor(device=device)
    
    # Use smaller batch size for CPU, larger for GPU
    batch_size = 64 if device == "cuda" else 16
    
    # Extract embeddings from training data
    print("  Processing training set...")
    X_train_bert = embedding_extractor.get_embeddings(
        train_df_text['text_combined'].tolist(),
        batch_size=batch_size
    )
    
    # Extract embeddings from validation data
    print("  Processing validation set...")
    X_val_bert = embedding_extractor.get_embeddings(
        val_df_text['text_combined'].tolist(),
        batch_size=batch_size
    )
    
    # Extract embeddings from test data
    print("  Processing test set...")
    X_test_bert = embedding_extractor.get_embeddings(
        test_df_text['text_combined'].tolist(),
        batch_size=batch_size
    )
    
    # Cache the embeddings for future runs
    print("\n💾 Caching embeddings for future use...")
    with open(cache_path, 'wb') as f:
        pickle.dump({
            'X_train_bert': X_train_bert,
            'X_val_bert': X_val_bert,
            'X_test_bert': X_test_bert
        }, f)
    print(f"   Saved to: {cache_path}")

# ============================================================
# Dimensionality Reduction: Reduce AraBERT (768) → 100 using PCA
# ============================================================
print("\n🔻 Reducing AraBERT embeddings dimensionality using PCA (768 → 100)...")
pca = PCA(n_components=100, random_state=42)
X_train_bert_reduced = pca.fit_transform(X_train_bert)
X_val_bert_reduced = pca.transform(X_val_bert)
X_test_bert_reduced = pca.transform(X_test_bert)
print(f"AraBERT reduced: {X_train_bert.shape[1]} → {X_train_bert_reduced.shape[1]} features")
print(f"Variance explained: {pca.explained_variance_ratio_.sum():.2%}")

# Combine numerical features with reduced semantic embeddings
X_train_with_text = np.hstack([X_train_scaled, X_train_bert_reduced])
X_val_with_text = np.hstack([X_val_scaled, X_val_bert_reduced])
X_test_with_text = np.hstack([X_test_scaled, X_test_bert_reduced])

print(f"\n✅ Semantic embeddings extracted and reduced (AraBERT):")
print(f"   Original numerical features: {X_train_scaled.shape[1]}")
print(f"   AraBERT (reduced) semantic features: {X_train_bert_reduced.shape[1]}")
print(f"   Total combined features: {X_train_with_text.shape[1]}")
print(f"\n💡 Note: AraBERT captures semantic meaning in MSA (Modern Standard Arabic).")
print(f"   Better suited for educational video titles than dialect models.")


## Step 6.5: Extract TF-IDF Features from Text

Add semantic features from video titles and descriptions using TF-IDF (Term Frequency-Inverse Document Frequency).
This captures topic-specific keywords that distinguish high-engagement videos from low-engagement ones.

In [ ]:
# ============================================================
# STEP 6.5: EXTRACT TF-IDF FEATURES FROM TEXT
# ============================================================

from sklearn.feature_extraction.text import TfidfVectorizer

print("📝 Extracting TF-IDF features from video titles and descriptions...")

# Prepare text data (same as used for AraBERT)
# FIX: previous code referenced train_df_text, val_df_text, test_df_text which may not be defined here.
# Let's use the same DataFrames already used for embedding extraction, which are train_df, val_df, test_df.
# Create text DataFrames from original df using the same indices
train_df_text = df.loc[train_idx, ['title', 'description']].copy()
val_df_text = df.loc[val_idx, ['title', 'description']].copy()
test_df_text = df.loc[test_idx, ['title', 'description']].copy()

# Combine title + description for richer context
train_df_text['text_combined'] = (
    train_df_text['title'].fillna('') + ' ' + 
    train_df_text['description'].fillna('')
)
val_df_text['text_combined'] = (
    val_df_text['title'].fillna('') + ' ' + 
    val_df_text['description'].fillna('')
)
test_df_text['text_combined'] = (
    test_df_text['title'].fillna('') + ' ' + 
    test_df_text['description'].fillna('')
)
train_text_combined = train_df_text['text_combined'].tolist()
val_text_combined = val_df_text['text_combined'].tolist()
test_text_combined = test_df_text['text_combined'].tolist()

# Initialize TF-IDF vectorizer
tfidf = TfidfVectorizer(
    max_features=30,      # Top 1500 most important terms
    min_df=10,               # Term must appear in at least 2 documents
    max_df=0.5,             # Ignore terms that appear in >80% of documents
    ngram_range=(1, 2),     # Unigrams and bigrams
    sublinear_tf=True       # Use log scaling for term frequency
)

# Fit on training data and transform all sets
print("  Processing training set...")
X_train_tfidf = tfidf.fit_transform(train_text_combined)

print("  Processing validation set...")
X_val_tfidf = tfidf.transform(val_text_combined)

print("  Processing test set...")
X_test_tfidf = tfidf.transform(test_text_combined)

# Convert sparse matrices to dense arrays for combining with other features
X_train_tfidf = X_train_tfidf.toarray()
X_val_tfidf = X_val_tfidf.toarray()
X_test_tfidf = X_test_tfidf.toarray()

print(f"\n✅ TF-IDF features extracted:")
print(f"   Vocabulary size: {len(tfidf.vocabulary_)}")
print(f"   Feature dimensions: {X_train_tfidf.shape[1]}")
print(f"   Sample terms: {list(tfidf.get_feature_names_out()[:30])}")

## Step 6.6: Combine All Features (Strategy 1)

**Improvement Strategy**: Combine TF-IDF + AraBERT + Numerical Features

### Why This Works

Based on research (documented in `docs/02_model_baseline_testing/arabert_implementation_guide.md`):

1. **TF-IDF captures discriminative keywords**: "بكالوريا", "رياضيات", specific subject terms
2. **AraBERT captures semantic meaning**: Understands context and relationships between concepts
3. **Numerical features capture engagement patterns**: Views, duration, like ratios, etc.

### Expected Results

| Configuration | Expected R² | Expected MAE |
|--------------|-------------|--------------|
| AraBERT only | 0.34-0.43 | 0.69-0.72 |
| TF-IDF only | 0.44-0.46 | 0.62-0.64 |
| **Combined (All 3)** | **0.48-0.52** | **0.58-0.62** |

The combination leverages the strengths of each feature type, leading to better overall performance than any single approach.

In [ ]:
# ============================================================
# STRATEGY 1: COMBINE ALL FEATURES (TF-IDF + AraBERT + NUMERICAL)
# Implementing improvement strategy from arabert_implementation_guide.md
# ============================================================

print("🔗 Combining TF-IDF + AraBERT + Numerical features...")
print("   This combines the strengths of:")
print("   - TF-IDF: Discriminative keywords (task-specific)")
print("   - AraBERT: Semantic understanding (context-aware)")
print("   - Numerical: Engagement metrics (views, duration, etc.)\n")

# Combine all three feature types
X_train_combined = np.hstack([
    X_train_scaled,      # 14 numerical features
    X_train_tfidf,       # 1500 TF-IDF features
    X_train_bert_reduced         # 768 AraBERT features
])

X_val_combined = np.hstack([
    X_val_scaled,
    X_val_tfidf,
    X_val_bert_reduced
])

X_test_combined = np.hstack([
    X_test_scaled,
    X_test_tfidf,
    X_test_bert_reduced
])

print(f"✅ Feature combination complete:")
print(f"   Numerical features:  {X_train_scaled.shape[1]:>4} dims")
print(f"   TF-IDF features:     {X_train_tfidf.shape[1]:>4} dims")
print(f"   AraBERT features:    {X_train_bert_reduced.shape[1]:>4} dims")
print(f"   {'─' * 35}")
print(f"   Total combined:      {X_train_combined.shape[1]:>4} dims")
print(f"\n💡 Expected improvement: R² 0.48-0.52 (vs 0.43 AraBERT-only)")

# Update X_train_with_text to use combined features for model training
X_train_with_text = X_train_combined
X_val_with_text = X_val_combined
X_test_with_text = X_test_combined

In [ ]:
# Define regression models
from sklearn.linear_model import Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

models = {
    'Ridge Regression': Ridge(alpha=1.0, random_state=42),
    'Lasso Regression': Lasso(alpha=1.0, random_state=42),
    'Random Forest': RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1, max_depth=20),
    # 'SVR': SVR(kernel='rbf'),  # SKIPPED: Extremely slow on large datasets, poor performance
    #'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42, max_depth=5)
}

# Add ensemble models if available (with early stopping for faster training)
if XGBOOST_AVAILABLE:
    from xgboost import XGBRegressor
    models['XGBoost'] = XGBRegressor(random_state=42, n_estimators=100, early_stopping_rounds=10, max_depth=5, device='cuda')
    
if LIGHTGBM_AVAILABLE:
    from lightgbm import LGBMRegressor
    models['LightGBM'] = LGBMRegressor(random_state=42, verbose=-1, n_estimators=100, max_depth=5, n_jobs=-1) 

print(f"🤖 Training {len(models)} baseline models:")
for model_name in models.keys():
    print(f"  - {model_name}")

In [ ]:
# Train and evaluate models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

results = {}

for model_name, model in models.items():
    print(f"\n🔧 Training {model_name}...")
    
    try:
        # PERFORMANCE: Use sparse matrices for Ridge/Lasso (much faster)
        if model_name in ['Ridge Regression', 'Lasso Regression']:
            model.fit(X_train_with_text, y_train)
            y_val_pred = model.predict(X_val_with_text)
        # Early stopping for XGBoost
        elif model_name == 'XGBoost' and XGBOOST_AVAILABLE:
            model.fit(
                X_train_with_text, y_train,
                eval_set=[(X_val_with_text, y_val)],
                verbose=False
            )
            y_val_pred = model.predict(X_val_with_text)
        # Early stopping for LightGBM
        elif model_name == 'LightGBM' and LIGHTGBM_AVAILABLE:
            model.fit(
                X_train_with_text, y_train,
                eval_set=[(X_val_with_text, y_val)]
            )
            y_val_pred = model.predict(X_val_with_text)
        else:
            # Train with TF-IDF features (dense arrays for other models)
            model.fit(X_train_with_text, y_train)
            y_val_pred = model.predict(X_val_with_text)
        
        # Calculate regression metrics
        mae = mean_absolute_error(y_val, y_val_pred)
        mse = mean_squared_error(y_val, y_val_pred)
        rmse = np.sqrt(mse)
        r2 = r2_score(y_val, y_val_pred)
        
        results[model_name] = {
            'model': model,
            'mae': mae,
            'mse': mse,
            'rmse': rmse,
            'r2_score': r2,
            'predictions': y_val_pred
        }
        
        print(f"  ✅ MAE: {mae:.4f} | RMSE: {rmse:.4f} | R²: {r2:.4f}")
        
    except Exception as e:
        print(f"  ❌ Error: {e}")
        continue

print("\n✅ All models trained!")

## Step 7.5: XGBoost Hyperparameter Tuning with Optuna

In [ ]:
# ============================================================
# OPTUNA XGBOOST WITH STRONGER REGULARIZATION
# ============================================================

import optuna
from xgboost import XGBRegressor
from sklearn.metrics import mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

def objective_regularized(trial):
    """Optuna objective with stronger regularization to reduce overfitting."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),
        'max_depth': trial.suggest_int('max_depth', 2, 6),  # Shallower trees
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.15, log=True),  # Lower max LR
        'subsample': trial.suggest_float('subsample', 0.5, 0.8),  # More dropout
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),  # More feature dropout
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 20),  # Higher = more conservative
        'reg_alpha': trial.suggest_float('reg_alpha', 0.1, 100.0, log=True),  # L1 regularization
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 100.0, log=True),  # L2 regularization
        'gamma': trial.suggest_float('gamma', 0.1, 10.0, log=True),  # Min loss reduction
        'random_state': RANDOM_SEED,
        'n_jobs': -1,
        'early_stopping_rounds': 20,  # Stop if no improvement
    }
    # Update X_train_with_text to use combined features for model training
    model = XGBRegressor(**params)
    model.fit(
        X_train_with_text, y_train, 
        eval_set=[(X_val_with_text, y_val)], 
        verbose=False
    )
    
    y_pred = model.predict(X_val_with_text)
    r2 = r2_score(y_val, y_pred)
    
    # Penalize overfitting: if train R² >> val R², reduce score
    train_r2 = r2_score(y_train, model.predict(X_train_with_text))
    overfit_penalty = max(0, (train_r2 - r2 - 0.1) * 0.5)  # Penalize if gap > 10%
    
    return r2 - overfit_penalty

# Run optimization
print("🔍 Starting Optuna search with STRONGER regularization (50 trials)...")
study_reg = optuna.create_study(direction='maximize', study_name='xgboost_regularized')
study_reg.optimize(objective_regularized, n_trials=50, show_progress_bar=True)

print(f"\n✅ Best R² on validation: {study_reg.best_value:.4f}")
print(f"📊 Best hyperparameters:")
for k, v in study_reg.best_params.items():
    print(f"   {k}: {v}")

In [ ]:
# Train final regularized model
print("🏆 Training final XGBoost model with regularization...")

# Remove early_stopping_rounds for final training (not needed with full data)
final_params = {k: v for k, v in study_reg.best_params.items() if k != 'early_stopping_rounds'}

best_xgb_reg = XGBRegressor(
    **final_params,
    random_state=RANDOM_SEED,
    n_jobs=-1
)

best_xgb_reg.fit(X_train_with_text, y_train)  # ← FIXED

# Evaluate on all sets
for name, X, y in [("Train", X_train_with_text, y_train),   # ← FIXED
                   ("Val", X_val_with_text, y_val),          # ← FIXED
                   ("Test", X_test_with_text, y_test)]:      # ← FIXED
    y_pred = best_xgb_reg.predict(X)
    mae = mean_absolute_error(y, y_pred)
    rmse = np.sqrt(mean_squared_error(y, y_pred))
    r2 = r2_score(y, y_pred)
    print(f"\n{name}:")
    print(f"   MAE: {mae:.4f}")
    print(f"   RMSE: {rmse:.4f}")
    print(f"   R²: {r2:.4f}")

# Check overfitting gap
train_r2 = r2_score(y_train, best_xgb_reg.predict(X_train_with_text))  # ← FIXED
val_r2 = r2_score(y_val, best_xgb_reg.predict(X_val_with_text))        # ← FIXED
print(f"\n📊 Overfitting gap: {train_r2 - val_r2:.4f}")

In [ ]:
# ============================================================
# CATBOOST WITH OPTUNA TUNING
# ============================================================

from catboost import CatBoostRegressor

def objective_catboost(trial):
    """Optuna objective for CatBoost."""
    params = {
        'iterations': trial.suggest_int('iterations', 100, 500),
        'depth': trial.suggest_int('depth', 4, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1.0, 50.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 1.0),
        'random_strength': trial.suggest_float('random_strength', 0.1, 10.0, log=True),
        'random_seed': RANDOM_SEED,
        'verbose': False,
    }
    
    model = CatBoostRegressor(**params)
    model.fit(X_train_with_text, y_train, eval_set=(X_val_with_text, y_val), early_stopping_rounds=30, verbose=False)  # ← FIXED
    
    y_pred = model.predict(X_val_with_text)  # ← FIXED
    return r2_score(y_val, y_pred)

print("🔍 Starting CatBoost Optuna search (30 trials)...")
study_cat = optuna.create_study(direction='maximize', study_name='catboost_tuning')
study_cat.optimize(objective_catboost, n_trials=30, show_progress_bar=True)

print(f"\n✅ Best CatBoost R² on validation: {study_cat.best_value:.4f}")

In [ ]:
# Train final CatBoost
best_cat = CatBoostRegressor(**study_cat.best_params, random_seed=RANDOM_SEED, verbose=False)
best_cat.fit(X_train_with_text, y_train)  # ← FIXED

print("🏆 CatBoost Results:")
for name, X, y in [("Train", X_train_with_text, y_train),   # ← FIXED
                   ("Val", X_val_with_text, y_val),          # ← FIXED
                   ("Test", X_test_with_text, y_test)]:      # ← FIXED
    y_pred = best_cat.predict(X)
    print(f"\n{name}: MAE={mean_absolute_error(y, y_pred):.4f}, R²={r2_score(y, y_pred):.4f}")

In [ ]:
# ============================================================
# NEURAL NETWORK (MLP) WITH PYTORCH
# ============================================================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🔧 Using device: {device}")

# Convert to tensors
X_train_t = torch.FloatTensor(X_train_with_text).to(device)  # ← FIXED
y_train_t = torch.FloatTensor(y_train).to(device)
X_val_t = torch.FloatTensor(X_val_with_text).to(device)      # ← FIXED
y_val_t = torch.FloatTensor(y_val).to(device)
X_test_t = torch.FloatTensor(X_test_with_text).to(device)    # ← FIXED
y_test_t = torch.FloatTensor(y_test).to(device)

# Define MLP
class EngagementMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.net(x).squeeze()

# Initialize model
input_dim = X_train_with_text.shape[1]  # ← FIXED
model = EngagementMLP(input_dim).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=20, factor=0.5)

# Training
train_dataset = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

print("🏋️ Training Neural Network...")
best_val_loss = float('inf')
patience_counter = 0

for epoch in range(200):
    model.train()
    train_loss = 0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        y_pred = model(X_batch)
        loss = criterion(y_pred, y_batch)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    
    # Validation
    model.eval()
    with torch.no_grad():
        val_pred = model(X_val_t)
        val_loss = criterion(val_pred, y_val_t).item()
        val_r2 = r2_score(y_val, val_pred.cpu().numpy())
    
    scheduler.step(val_loss)
    
    # Early stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_model_state = model.state_dict().copy()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= 20:
            print(f"   Early stopping at epoch {epoch}")
            break
    
    if epoch % 20 == 0:
        print(f"   Epoch {epoch}: Train Loss={train_loss/len(train_loader):.4f}, Val R²={val_r2:.4f}")

# Load best model
model.load_state_dict(best_model_state)

In [ ]:
# Evaluate NN
model.eval()
print("\n🏆 Neural Network Results:")
with torch.no_grad():
    for name, X_t, y in [("Train", X_train_t, y_train), 
                         ("Val", X_val_t, y_val), 
                         ("Test", X_test_t, y_test)]:
        y_pred = model(X_t).cpu().numpy()
        mae = mean_absolute_error(y, y_pred)
        r2 = r2_score(y, y_pred)
        print(f"{name}: MAE={mae:.4f}, R²={r2:.4f}")

## 7.6 ensambling


In [ ]:
# ============================================================
# MODEL ENSEMBLE (WEIGHTED AVERAGING)
# ============================================================

from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

print("🔗 Building Model Ensemble...")

# Retrain top 3 models for ensemble (if not already trained)
# Random Forest
rf_ensemble = RandomForestRegressor(n_estimators=200, max_depth=15, random_state=RANDOM_SEED, n_jobs=-1)
rf_ensemble.fit(X_train_with_text, y_train)          # ← FIXED
pred_rf_val = rf_ensemble.predict(X_val_with_text)    # ← FIXED
pred_rf_test = rf_ensemble.predict(X_test_with_text)  # ← FIXED

# LightGBM
lgbm_ensemble = LGBMRegressor(n_estimators=200, max_depth=8, learning_rate=0.05, random_state=RANDOM_SEED, n_jobs=-1, verbose=-1)
lgbm_ensemble.fit(X_train_with_text, y_train)           # ← FIXED
pred_lgbm_val = lgbm_ensemble.predict(X_val_with_text)   # ← FIXED
pred_lgbm_test = lgbm_ensemble.predict(X_test_with_text) # ← FIXED

# XGBoost
xgb_ensemble = XGBRegressor(n_estimators=200, max_depth=6, learning_rate=0.05, random_state=RANDOM_SEED, n_jobs=-1)
xgb_ensemble.fit(X_train_with_text, y_train)           # ← FIXED
pred_xgb_val = xgb_ensemble.predict(X_val_with_text)   # ← FIXED
pred_xgb_test = xgb_ensemble.predict(X_test_with_text) # ← FIXED

# CatBoost (if available)
if 'best_cat' in dir():
    pred_cat_val = best_cat.predict(X_val_with_text)   # ← FIXED
    pred_cat_test = best_cat.predict(X_test_with_text) # ← FIXED
else:
    pred_cat_val = pred_xgb_val
    pred_cat_test = pred_xgb_test

# ============================================================
# Try different ensemble weights
# ============================================================

print("\n📊 Testing different ensemble combinations:\n")

ensemble_configs = [
    ("Equal (RF+LGBM+XGB)", [1/3, 1/3, 1/3, 0]),
    ("RF-heavy (50% RF)", [0.5, 0.25, 0.25, 0]),
    ("Top 2 (RF+LGBM)", [0.5, 0.5, 0, 0]),
    ("All 4 models", [0.35, 0.25, 0.20, 0.20]),
]

best_ensemble_r2 = 0
best_ensemble_name = ""

for name, weights in ensemble_configs:
    pred_val = (weights[0] * pred_rf_val + 
                weights[1] * pred_lgbm_val + 
                weights[2] * pred_xgb_val + 
                weights[3] * pred_cat_val)
    
    pred_test = (weights[0] * pred_rf_test + 
                 weights[1] * pred_lgbm_test + 
                 weights[2] * pred_xgb_test + 
                 weights[3] * pred_cat_test)
    
    val_r2 = r2_score(y_val, pred_val)
    test_r2 = r2_score(y_test, pred_test)
    val_mae = mean_absolute_error(y_val, pred_val)
    
    print(f"{name:25s} | Val R²: {val_r2:.4f} | Test R²: {test_r2:.4f} | Val MAE: {val_mae:.4f}")
    
    if val_r2 > best_ensemble_r2:
        best_ensemble_r2 = val_r2
        best_ensemble_name = name
        best_pred_val = pred_val
        best_pred_test = pred_test

print(f"\n🏆 Best Ensemble: {best_ensemble_name} (Val R² = {best_ensemble_r2:.4f})")

# Add best ensemble to results
results['Ensemble (' + best_ensemble_name + ')'] = {
    'mae': mean_absolute_error(y_val, best_pred_val),
    'rmse': np.sqrt(mean_squared_error(y_val, best_pred_val)),
    'r2_score': best_ensemble_r2
}

# Final comparison
print(f"\n📈 Improvement over single Random Forest:")
print(f"   RF alone: R² = 0.5921")
print(f"   Ensemble: R² = {best_ensemble_r2:.4f}")
print(f"   Gain: {(best_ensemble_r2 - 0.5921)*100:.2f}%")

In [ ]:
# ============================================================
# OPTUNA HYPERPARAMETER TUNING FOR RANDOM FOREST
# ============================================================

import optuna

def objective_rf(trial):
    """Optuna objective for Random Forest."""
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 400),
        'max_depth': trial.suggest_int('max_depth', 8, 25),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.5, 0.7]),
        'random_state': RANDOM_SEED,
        'n_jobs': -1
    }
    
    model = RandomForestRegressor(**params)
    model.fit(X_train_with_text, y_train)  # ← FIXED
    
    y_pred = model.predict(X_val_with_text)  # ← FIXED
    return r2_score(y_val, y_pred)

print("🔍 Tuning Random Forest with Optuna (40 trials)...")
study_rf = optuna.create_study(direction='maximize', study_name='rf_tuning')
study_rf.optimize(objective_rf, n_trials=40, show_progress_bar=True)

print(f"\n✅ Best R² on validation: {study_rf.best_value:.4f}")
print(f"📊 Best hyperparameters:")
for k, v in study_rf.best_params.items():
    print(f"   {k}: {v}")

In [ ]:
# Train final RF with best parameters
best_rf = RandomForestRegressor(**study_rf.best_params, random_state=RANDOM_SEED, n_jobs=-1)
best_rf.fit(X_train_with_text, y_train)  # ← FIXED

print("🏆 Random Forest (Optuna-Tuned) Results:")
for name, X, y in [("Train", X_train_with_text, y_train),   # ← FIXED
                   ("Val", X_val_with_text, y_val),          # ← FIXED
                   ("Test", X_test_with_text, y_test)]:      # ← FIXED
    y_pred = best_rf.predict(X)
    r2 = r2_score(y, y_pred)
    mae = mean_absolute_error(y, y_pred)
    print(f"   {name}: R²={r2:.4f}, MAE={mae:.4f}")

# Add to results
results['Random Forest (Optuna)'] = {
    'mae': mean_absolute_error(y_val, best_rf.predict(X_val_with_text)),   # ← FIXED
    'rmse': np.sqrt(mean_squared_error(y_val, best_rf.predict(X_val_with_text))),  # ← FIXED
    'r2_score': study_rf.best_value
}

## Step 8: Model Performance Comparison

In [ ]:
# ============================================================
# COMPREHENSIVE MODEL COMPARISON (ALL MODELS)
# ============================================================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Determine which feature set each model was trained on
# Models trained on X_with_text need X_val_with_text for prediction

# Add all new models to results dictionary
# XGBoost (Optuna - original)
if 'best_xgb' in dir():
    results['XGBoost (Optuna)'] = {
        'mae': mean_absolute_error(y_val, best_xgb.predict(X_val_with_text)),
        'rmse': np.sqrt(mean_squared_error(y_val, best_xgb.predict(X_val_with_text))),
        'r2_score': r2_score(y_val, best_xgb.predict(X_val_with_text))
    }

# XGBoost (Regularized)
if 'best_xgb_reg' in dir():
    results['XGBoost (Regularized)'] = {
        'mae': mean_absolute_error(y_val, best_xgb_reg.predict(X_val_with_text)),
        'rmse': np.sqrt(mean_squared_error(y_val, best_xgb_reg.predict(X_val_with_text))),
        'r2_score': r2_score(y_val, best_xgb_reg.predict(X_val_with_text))
    }

# CatBoost
if 'best_cat' in dir():
    results['CatBoost (Optuna)'] = {
        'mae': mean_absolute_error(y_val, best_cat.predict(X_val_with_text)),
        'rmse': np.sqrt(mean_squared_error(y_val, best_cat.predict(X_val_with_text))),
        'r2_score': r2_score(y_val, best_cat.predict(X_val_with_text))
    }

# Neural Network
if 'model' in dir() and hasattr(model, 'eval'):
    model.eval()
    # Neural network needs the correct input tensor
    X_val_nn = torch.tensor(X_val_with_text, dtype=torch.float32).to(device)
    with torch.no_grad():
        nn_pred = model(X_val_nn).cpu().numpy()
    results['Neural Network (MLP)'] = {
        'mae': mean_absolute_error(y_val, nn_pred),
        'rmse': np.sqrt(mean_squared_error(y_val, nn_pred)),
        'r2_score': r2_score(y_val, nn_pred)
    }

# Create comprehensive comparison dataframe
comparison_df = pd.DataFrame({
    'Model': list(results.keys()),
    'MAE': [results[m]['mae'] for m in results.keys()],
    'RMSE': [results[m]['rmse'] for m in results.keys()],
    'R²': [results[m]['r2_score'] for m in results.keys()]
})

# Sort by R² (higher is better)
comparison_df = comparison_df.sort_values('R²', ascending=False).reset_index(drop=True)

print("=" * 70)
print("📊 COMPREHENSIVE MODEL COMPARISON (Validation Set)")
print("=" * 70)
print(comparison_df.to_string(index=False))
print("=" * 70)

# Highlight best model
best_model_name = comparison_df.iloc[0]['Model']
best_r2 = comparison_df.iloc[0]['R²']
print(f"\n🏆 Best Model: {best_model_name} (R² = {best_r2:.4f})")

# Show improvement potential
baseline_r2 = comparison_df[comparison_df['Model'] == 'Random Forest']['R²'].values[0] if 'Random Forest' in comparison_df['Model'].values else 0
print(f"\n📈 Target R² = 0.70 | Current Best = {best_r2:.4f} | Gap = {0.70 - best_r2:.4f}")

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = np.arange(len(comparison_df))

# MAE (lower is better)
axes[0].bar(x, comparison_df['MAE'], color='coral', alpha=0.8)
axes[0].set_xlabel('Model')
axes[0].set_ylabel('Mean Absolute Error')
axes[0].set_title('MAE (Lower is Better)')
axes[0].set_xticks(x)
axes[0].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# RMSE (lower is better)
axes[1].bar(x, comparison_df['RMSE'], color='steelblue', alpha=0.8)
axes[1].set_xlabel('Model')
axes[1].set_ylabel('Root Mean Squared Error')
axes[1].set_title('RMSE (Lower is Better)')
axes[1].set_xticks(x)
axes[1].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[1].grid(axis='y', alpha=0.3)

# R² (higher is better)
axes[2].bar(x, comparison_df['R²'], color='seagreen', alpha=0.8)
axes[2].set_xlabel('Model')
axes[2].set_ylabel('R² Score')
axes[2].set_title('R² Score (Higher is Better)')
axes[2].set_xticks(x)
axes[2].set_xticklabels(comparison_df['Model'], rotation=45, ha='right')
axes[2].grid(axis='y', alpha=0.3)
axes[2].axhline(y=0, color='red', linestyle='--', linewidth=1)

plt.tight_layout()
plt.show()

## Step 9: Detailed Analysis of Best Model

In [ ]:
# Get best model and generate predictions
# The models are stored in separate variables, not in the results dict

print(f"📊 Detailed Metrics for {best_model_name}:")
print(f"  MAE:  {results[best_model_name]['mae']:.4f}")
print(f"  RMSE: {results[best_model_name]['rmse']:.4f}")
print(f"  R²:   {results[best_model_name]['r2_score']:.4f}")

# Get predictions based on best model name
if 'XGBoost (Regularized)' in best_model_name:
    y_val_pred = best_xgb_reg.predict(X_val_with_text)
elif 'XGBoost (Optuna)' in best_model_name:
    y_val_pred = best_xgb.predict(X_val_with_text)
elif 'CatBoost' in best_model_name:
    y_val_pred = best_cat.predict(X_val_with_text)
elif 'Neural Network' in best_model_name:
    model.eval()
    X_val_nn = torch.tensor(X_val_with_text, dtype=torch.float32).to(device)
    with torch.no_grad():
        y_val_pred = model(X_val_nn).cpu().numpy().flatten()
elif 'Random Forest' in best_model_name:
    y_val_pred = best_rf.predict(X_val_with_text) if 'best_rf' in dir() else rf.predict(X_val_with_text)
else:
    # Fallback - try to find the model
    print(f"⚠️ Unknown model: {best_model_name}")
    y_val_pred = None

# Error analysis
if y_val_pred is not None:
    residuals = y_val - y_val_pred
    print(f"\n📈 Error Distribution:")
    print(f"  Mean residual: {residuals.mean():.4f}")
    print(f"  Std residual:  {residuals.std():.4f}")
    print(f"  Min residual:  {residuals.min():.4f}")
    print(f"  Max residual:  {residuals.max():.4f}")

In [ ]:
# Scatter plot: Predicted vs Actual
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Predicted vs Actual
axes[0].scatter(y_val, y_val_pred, alpha=0.5, s=20)
axes[0].plot([y_val.min(), y_val.max()], [y_val.min(), y_val.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Engagement Score')
axes[0].set_ylabel('Predicted Engagement Score')
axes[0].set_title(f'Predicted vs Actual - {best_model_name}')
axes[0].legend()
axes[0].grid(alpha=0.3)


# Right: Residual plot

residuals = y_val - y_val_pred

axes[1].scatter(y_val_pred, residuals, alpha=0.5, s=20)

axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].grid(alpha=0.3)

axes[1].set_xlabel('Predicted Engagement Score')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title(f'Residual Plot - {best_model_name}')

plt.tight_layout()
plt.show()
axes[1].set_ylabel('Residuals (Actual - Predicted)')

In [ ]:
# Get the actual model object based on best_model_name
if 'XGBoost (Regularized)' in best_model_name:
    best_model = best_xgb_reg
elif 'XGBoost (Optuna)' in best_model_name:
    best_model = best_xgb
elif 'CatBoost' in best_model_name:
    best_model = best_cat
elif 'Random Forest' in best_model_name:
    best_model = best_rf if 'best_rf' in dir() else rf
elif 'Neural Network' in best_model_name:
    best_model = model  # Neural network doesn't have feature_importances_
else:
    print(f"⚠️ Unknown model: {best_model_name}")
    best_model = None

print(f"🎯 Analyzing: {best_model_name}")

## Step 10: Feature Importance Analysis

In [ ]:
# Extract feature importance (if available)
# Check if TF-IDF features were used (model was trained with X_train_with_text)
try:
    # Check if TF-IDF features exist
    if 'tfidf' in globals() and 'X_train_with_text' in globals():
        # Model includes TF-IDF features - combine feature names
        numerical_feature_names = encoded_feature_cols
        tfidf_feature_names = [f"tfidf_{term}" for term in tfidf.get_feature_names_out()]
        all_feature_names = numerical_feature_names + tfidf_feature_names
    else:
        # No TF-IDF features - use only numerical features
        all_feature_names = encoded_feature_cols
except NameError:
    # TF-IDF not defined - use only numerical features
    all_feature_names = encoded_feature_cols

if hasattr(best_model, 'feature_importances_'):
    # Tree-based models
    importances = best_model.feature_importances_
    
    # Verify feature count matches
    if len(importances) != len(all_feature_names):
        print(f"⚠️ Warning: Feature count mismatch!")
        print(f"   Model has {len(importances)} features")
        print(f"   Feature names list has {len(all_feature_names)} features")
        print(f"   Using first {min(len(importances), len(all_feature_names))} features")
        min_len = min(len(importances), len(all_feature_names))
        importances = importances[:min_len]
        all_feature_names = all_feature_names[:min_len]
    
    feature_importance_df = pd.DataFrame({
        'feature': all_feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(f"📊 Top 15 Most Important Features ({best_model_name}):")
    print(feature_importance_df.head(15).to_string(index=False))
    
    # Plot
    plt.figure(figsize=(10, 8))
    top_n = 20
    top_features = feature_importance_df.head(top_n)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Importance')
    plt.title(f'Top {top_n} Feature Importances - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
elif hasattr(best_model, 'coef_'):
    # Linear models
    importances = np.abs(best_model.coef_).mean(axis=0)
    
    # Verify feature count matches
    if len(importances) != len(all_feature_names):
        print(f"⚠️ Warning: Feature count mismatch!")
        print(f"   Model has {len(importances)} features")
        print(f"   Feature names list has {len(all_feature_names)} features")
        print(f"   Using first {min(len(importances), len(all_feature_names))} features")
        min_len = min(len(importances), len(all_feature_names))
        importances = importances[:min_len]
        all_feature_names = all_feature_names[:min_len]
    
    feature_importance_df = pd.DataFrame({
        'feature': all_feature_names,
        'importance': importances
    }).sort_values('importance', ascending=False)
    
    print(f"📊 Top 15 Most Important Features ({best_model_name}):")
    print(feature_importance_df.head(15).to_string(index=False))
    
    # Plot
    plt.figure(figsize=(10, 8))
    top_n = 20
    top_features = feature_importance_df.head(top_n)
    plt.barh(range(len(top_features)), top_features['importance'])
    plt.yticks(range(len(top_features)), top_features['feature'])
    plt.xlabel('Absolute Coefficient Value')
    plt.title(f'Top {top_n} Feature Importances - {best_model_name}')
    plt.gca().invert_yaxis()
    plt.tight_layout()
    plt.show()
    
else:
    print(f"⚠️ {best_model_name} does not support feature importance extraction")

In [ ]:
# Updated categorization
def categorize_feature_v2(name):
    name_lower = name.lower()
    if name_lower.startswith('tfidf_'):
        return '📄 TF-IDF Keywords'
    elif any(x in name_lower for x in ['transcript', 'speech', 'flesch', 'fog', 'readability', 'sentence', 'lexical', 'unique_word']):
        return '📝 Transcript Quality'
    elif 'subject_' in name_lower:
        return '📚 Subject (One-Hot)'
    elif any(x in name_lower for x in ['bac', 'pedagogical', 'exam', 'domain_keyword']):
        return '🎓 Educational Markers'
    elif any(x in name_lower for x in ['title', 'description', 'question', 'example', 'explanation', 'contrast', 'technical']):
        return '✍️ Text Features'
    elif any(x in name_lower for x in ['duration', 'tag', 'weekday']):
        return '📹 Video Metadata'
    else:
        return '🔧 Other'
# Re-run category analysis
feature_importance_df['Category'] = feature_importance_df['feature'].apply(categorize_feature_v2)
category_importance = feature_importance_df.groupby('Category')['importance'].sum().sort_values(ascending=False)
print("📊 CORRECTED IMPORTANCE BY CATEGORY:")
print("=" * 60)
total = category_importance.sum()
for cat, imp in category_importance.items():
    pct = imp / total * 100
    bar = "█" * int(pct)
    print(f"{cat:25s} {pct:5.1f}% {bar}")

# Pie chart
fig, ax = plt.subplots(figsize=(8, 8))
colors = plt.cm.Set3(np.linspace(0, 1, len(category_importance)))
wedges, texts, autotexts = ax.pie(
    category_importance, labels=category_importance.index, 
    autopct='%1.1f%%', startangle=90, colors=colors
)
ax.set_title('Feature Importance by Category', fontsize=14)
plt.tight_layout()
plt.savefig(DATA_DIR / 'modeling' / 'feature_importance_categories.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\n💾 Saved: {DATA_DIR / 'modeling' / 'feature_importance_categories.png'}")

In [ ]:
# ============================================================
# INVESTIGATE "OTHER" CATEGORY
# ============================================================

other_features = feature_importance_df[feature_importance_df['Category'] == '🔧 Other']
print("📊 Features in 'Other' Category:")
print("=" * 60)
for idx, row in other_features.iterrows():
    print(f"   {row['feature']:40s} {row['importance']:.4f}")

print(f"\n📈 Total 'Other' importance: {other_features['importance'].sum():.4f}")
print(f"   This is {other_features['importance'].sum() / feature_importance_df['importance'].sum() * 100:.1f}% of total")

In [ ]:
# ============================================================
# KEY INSIGHTS SUMMARY
# ============================================================

print("\n" + "=" * 60)
print("🎯 KEY INSIGHTS FOR EDUCATIONAL VIDEO ENGAGEMENT")
print("=" * 60)

top_5 = feature_importance_df.head(5)['feature'].tolist()
print(f"\n📌 Top 5 Most Important Features:")
for i, f in enumerate(top_5, 1):
    print(f"   {i}. {f}")

# Category ranking
print(f"\n📊 Feature Category Ranking:")
for i, (cat, pct) in enumerate(category_importance.items(), 1):
    print(f"   {i}. {cat}: {pct/total*100:.1f}%")

print(f"\n💡 Actionable Recommendations for Content Creators:")
print("   • Optimize video DURATION for your subject")
print("   • Include clear EDUCATIONAL markers (بكالوريا, examen, etc.)")
print("   • Improve TRANSCRIPT quality (clear speech, good pacing)")
print("   • Use descriptive TITLES with exam keywords")
print("   • Subject choice matters for engagement potential")

In [ ]:
# Analyze TF-IDF Feature Contribution
if hasattr(best_model, 'feature_importances_'):
    # Get all feature names (numerical + TF-IDF)
    numerical_feature_names = encoded_feature_cols
    tfidf_feature_names = [f"tfidf_{term}" for term in tfidf.get_feature_names_out()]
    all_feature_names = numerical_feature_names + tfidf_feature_names

    # Get importances
    importances = best_model.feature_importances_
    
    # --- Fix: Ensure all arrays are the same length ---
    len_names = len(all_feature_names)
    len_importances = len(importances)
    len_types = len(numerical_feature_names) + len(tfidf_feature_names)
    min_len = min(len_names, len_importances, len_types)
    if len_names != len_importances or len_names != len_types:
        print(f"⚠️ Array length mismatch in feature importance analysis:")
        print(f"   - all_feature_names: {len_names}")
        print(f"   - importances: {len_importances}")
        print(f"   - types: {len_types}")
        print(f"   => Truncating all to {min_len} elements for consistency.")
    all_feature_names = all_feature_names[:min_len]
    importances = importances[:min_len]
    type_labels = (['numerical']*len(numerical_feature_names) + ['tfidf']*len(tfidf_feature_names))[:min_len]

    # Create dataframe
    feature_df = pd.DataFrame({
        'feature': all_feature_names,
        'importance': importances,
        'type': type_labels
    }).sort_values('importance', ascending=False)

    # Show top TF-IDF features
    top_tfidf = feature_df[feature_df['type'] == 'tfidf'].head(15)
    print("📊 Top 15 TF-IDF Features by Importance:")
    print(top_tfidf[['feature', 'importance']].to_string(index=False))

    # Compare contribution: numerical vs text
    numerical_contrib = feature_df[feature_df['type'] == 'numerical']['importance'].sum()
    tfidf_contrib = feature_df[feature_df['type'] == 'tfidf']['importance'].sum()

    print(f"\n📈 Feature Type Contribution:")
    if numerical_contrib + tfidf_contrib > 0:
        print(f"   Numerical features: {numerical_contrib:.3f} ({numerical_contrib/(numerical_contrib+tfidf_contrib)*100:.1f}%)")
        print(f"   TF-IDF features: {tfidf_contrib:.3f} ({tfidf_contrib/(numerical_contrib+tfidf_contrib)*100:.1f}%)")
    else:
        print(f"   Numerical features: {numerical_contrib:.3f} (0.0%)")
        print(f"   TF-IDF features: {tfidf_contrib:.3f} (0.0%)")

    # Plot comparison
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Left: Top 20 overall features
    top_20_all = feature_df.head(20)
    colors = ['steelblue' if t == 'numerical' else 'coral' for t in top_20_all['type']]
    ax1.barh(range(len(top_20_all)), top_20_all['importance'], color=colors)
    ax1.set_yticks(range(len(top_20_all)))
    ax1.set_yticklabels(top_20_all['feature'], fontsize=9)
    ax1.set_xlabel('Importance')
    ax1.set_title(f'Top 20 Most Important Features (All Types)')
    ax1.invert_yaxis()
    ax1.legend([plt.Rectangle((0,0),1,1, fc='steelblue'), 
                plt.Rectangle((0,0),1,1, fc='coral')], 
               ['Numerical', 'TF-IDF'], loc='lower right')

    # Right: Feature type contribution pie chart
    ax2.pie([numerical_contrib, tfidf_contrib], 
            labels=['Numerical', 'TF-IDF'],
            autopct='%1.1f%%',
            colors=['steelblue', 'coral'],
            startangle=90)
    ax2.set_title('Feature Importance Contribution by Type')

    plt.tight_layout()
    plt.show()

else:
    print(f"⚠️ {best_model_name} does not support feature importance analysis")

## Step 10.5: Analyze TF-IDF Feature Contribution

Check which text features are most important and compare their contribution vs numerical features.

In [ ]:
# Check components
print(f"Numerical: {X_test_scaled.shape[1]}")
print(f"AraBERT:   {X_test_bert_reduced.shape[1]}")
if 'X_test_tfidf' in dir():
    print(f"TF-IDF:    {X_test_tfidf.shape[1]}")

## Step 11: Test Set Evaluation

In [ ]:
# Evaluate best model on test set (using TF-IDF features)
y_test_pred = best_model.predict(X_test_with_text)

# Calculate regression metrics
test_mae = mean_absolute_error(y_test, y_test_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
test_rmse = np.sqrt(test_mse)
test_r2 = r2_score(y_test, y_test_pred)

print(f"🎯 {best_model_name} Performance on Test Set:")
print(f"  MAE:  {test_mae:.4f}")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

# Error analysis
test_residuals = y_test - y_test_pred
print(f"\n📈 Test Set Error Distribution:")
print(f"  Mean residual: {test_residuals.mean():.4f}")
print(f"  Std residual:  {test_residuals.std():.4f}")
print(f"  Min residual:  {test_residuals.min():.4f}")
print(f"  Max residual:  {test_residuals.max():.4f}")

In [ ]:
# Test set visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: Predicted vs Actual
axes[0].scatter(y_test, y_test_pred, alpha=0.5, s=20, color='green')
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Engagement Score')
axes[0].set_ylabel('Predicted Engagement Score')
axes[0].set_title(f'Test Set: Predicted vs Actual - {best_model_name}')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Right: Residual plot (DO NOT call plt.show() before this!)
test_residuals = y_test - y_test_pred
axes[1].scatter(y_test_pred, test_residuals, alpha=0.5, s=20, color='green')
axes[1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[1].set_xlabel('Predicted Engagement Score')
axes[1].set_ylabel('Residuals (Actual - Predicted)')
axes[1].set_title(f'Test Set: Residual Plot - {best_model_name}')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()  # Only ONE plt.show() at the end!

## Step 12: Save Models and Results

In [ ]:
import joblib

# Create models directory
models_dir = PROJECT_ROOT / "models"
models_dir.mkdir(parents=True, exist_ok=True)

# Save best model
model_path = models_dir / f"best_{best_model_name.lower().replace(' ', '_')}_baseline.pkl"
joblib.dump(best_model, model_path)
print(f"💾 Saved best model: {model_path}")

# Save scaler
scaler_path = models_dir / "scaler_baseline.pkl"
joblib.dump(scaler, scaler_path)
print(f"💾 Saved scaler: {scaler_path}")

# Save TF-IDF vectorizer (NEW!)
tfidf_path = models_dir / "tfidf_vectorizer.pkl"
joblib.dump(tfidf, tfidf_path)
print(f"💾 Saved TF-IDF vectorizer: {tfidf_path}")

# Save results
results_df = comparison_df.copy()
results_df.to_csv(modeling_dir / "baseline_results.csv", index=False)
print(f"💾 Saved results: {modeling_dir / 'baseline_results.csv'}")

# Save test performance
test_results = {
    'model': best_model_name,
    'task': 'regression',
    'target': 'engagement_score',
    'test_mae': test_mae,
    'test_mse': test_mse,
    'test_rmse': test_rmse,
    'test_r2': test_r2,
    'evaluated_at': datetime.now().isoformat(),
    'features': {
        'numerical': len(encoded_feature_cols),
        'tfidf': X_train_tfidf.shape[1],
        'arabert': X_train_bert_reduced.shape[1],
        'total': X_train_with_text.shape[1],
        'strategy': 'combined_all'  # TF-IDF + AraBERT + Numerical
    }
}

with open(modeling_dir / "test_results.json", 'w') as f:
    json.dump(test_results, f, indent=2)
print(f"💾 Saved test results: {modeling_dir / 'test_results.json'}")

print("\n✅ All artifacts saved!")

## Summary and Next Steps

### Key Findings
- **Best Model**: Check cell above for the winning model
- **Validation R² Score**: See comparison table
- **Test R² Score**: See test evaluation
- **Task Type**: Regression (predicting continuous engagement_score)

### Data Splits Saved
All splits have been saved to `data/modeling/`:
- ✅ `train.csv` - Training data
- ✅ `val.csv` - Validation data
- ✅ `test.csv` - Test data
- ✅ `splits_mapping.csv` - Channel to split assignment
- ✅ `split_metadata.json` - Split metadata and configuration

### Next Steps
1. **Hyperparameter Tuning**: Use GridSearchCV or RandomizedSearchCV on best model
3. **Ensemble Methods**: Try stacking or voting regressors
4. **Error Analysis**: Investigate videos with high prediction errors
5. **Cross-Validation**: Implement k-fold CV to validate results
6. **Outlier Analysis**: Examine videos with extreme residuals

### Notes
- Group-based splitting by `channel_id` prevents data leakage
- All videos from the same channel stay in the same split
- Feature scaling applied using StandardScaler

- Models saved in `models/` directory for future use- Models saved in `models/` directory for future use

## Step 13: Time-Series Validation (Production-Ready)
This validates model performance on FUTURE videos (realistic production scenario)

In [ ]:
# Add this cell in your notebook to analyze your target
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
ytrain_orig= np.expm1(y_train)
print("📊 Current Engagement Score Analysis:")
print(f"   Skewness: {pd.Series(ytrain_orig).skew():.4f}")
print(f"   Kurtosis: {pd.Series(ytrain_orig).kurtosis():.4f}")
print(f"   Min: {ytrain_orig.min():.4f}")
print(f"   Max: {ytrain_orig.max():.4f}")
# Shapiro-Wilk test for normality (sample of 5000 for speed)
sample = np.random.choice(ytrain_orig, size=min(5000, len(ytrain_orig)), replace=False)
stat, p_value = stats.shapiro(sample)
print(f"   Shapiro-Wilk p-value: {p_value:.4f}")
print(f"   Is Normal? {'Yes' if p_value > 0.05 else 'No'}")

# Plot distribution
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(ytrain_orig, bins=50, edgecolor='black')
axes[0].set_title('Current Target Distribution')
axes[0].set_xlabel('engagement_score')

stats.probplot(ytrain_orig, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (should be straight line if normal)')
plt.tight_layout()
plt.show()

In [ ]:
from scipy.stats import boxcox

# Test Box-Cox on your ACTUAL y_train (already log-transformed)
SHIFT = 0.01
y_train_shifted = y_train + SHIFT
y_train_bc, lambda_opt = boxcox(y_train_shifted)

print(f"📊 Box-Cox on Log-Transformed Target:")
print(f"   Optimal λ: {lambda_opt:.4f}")
print(f"   ")
print(f"   Current y_train skewness: {pd.Series(y_train).skew():.4f}")
print(f"   After Box-Cox skewness:   {pd.Series(y_train_bc).skew():.4f}")
print(f"   ")

# Interpretation
if abs(lambda_opt - 1) < 0.2:
    print("   → λ ≈ 1: Your log1p() is already near-optimal!")
    print("   → Box-Cox won't help much.")
else:
    print(f"   → λ = {lambda_opt:.2f}: Box-Cox found a better transform.")
    print("   → Worth trying in your model!")

In [ ]:
# ============================================================
# SMARTER STACKING: Use model diversity, not more features
# ============================================================

from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score, mean_absolute_error
import numpy as np

print("🔗 Building Diverse Stacking Ensemble...")

# Use your ORIGINAL features (without interactions)
# Step 1: Train diverse base models
base_models = {
    'rf': RandomForestRegressor(n_estimators=200, max_depth=15, random_state=42, n_jobs=-1),
    'gb': GradientBoostingRegressor(n_estimators=150, max_depth=5, learning_rate=0.1, random_state=42),
    'ridge': Ridge(alpha=1.0),
}

# Get out-of-fold predictions (prevents leakage)
print("\n   Generating out-of-fold predictions...")
meta_train = np.zeros((len(X_train_scaled), len(base_models)))
meta_val = np.zeros((len(X_val_scaled), len(base_models)))
meta_test = np.zeros((len(X_test_scaled), len(base_models)))

for i, (name, model) in enumerate(base_models.items()):
    print(f"   • {name}...", end=" ")
    
    # Out-of-fold predictions for training
    meta_train[:, i] = cross_val_predict(model, X_train_scaled, y_train, cv=5)
    
    # Fit on full training data for val/test
    model.fit(X_train_scaled, y_train)
    meta_val[:, i] = model.predict(X_val_scaled)
    meta_test[:, i] = model.predict(X_test_scaled)
    
    val_r2 = r2_score(y_val, meta_val[:, i])
    print(f"Val R² = {val_r2:.4f}")

# Step 2: Simple averaging (often works better than meta-learner)
y_pred_avg_val = meta_val.mean(axis=1)
y_pred_avg_test = meta_test.mean(axis=1)

print(f"\n{'='*55}")
print(f"📈 SIMPLE AVERAGE ENSEMBLE:")
print(f"{'='*55}")
print(f"   Validation R²: {r2_score(y_val, y_pred_avg_val):.4f}")
print(f"   Test R²:       {r2_score(y_test, y_pred_avg_test):.4f}")

# Step 3: Optimized weighted average
from scipy.optimize import minimize

def neg_r2(weights):
    pred = np.dot(meta_val, weights)
    return -r2_score(y_val, pred)

# Find optimal weights
result = minimize(neg_r2, x0=[1/3, 1/3, 1/3], 
                  bounds=[(0, 1)]*3,
                  constraints={'type': 'eq', 'fun': lambda w: sum(w) - 1})
optimal_weights = result.x

print(f"\n   Optimal weights: RF={optimal_weights[0]:.2f}, GB={optimal_weights[1]:.2f}, Ridge={optimal_weights[2]:.2f}")

y_pred_opt_val = np.dot(meta_val, optimal_weights)
y_pred_opt_test = np.dot(meta_test, optimal_weights)

print(f"\n{'='*55}")
print(f"📈 OPTIMIZED WEIGHTED ENSEMBLE:")
print(f"{'='*55}")
print(f"   Validation R²: {r2_score(y_val, y_pred_opt_val):.4f}")
print(f"   Test R²:       {r2_score(y_test, y_pred_opt_test):.4f}")
print(f"   Test MAE:      {mean_absolute_error(y_test, y_pred_opt_test):.4f}")
print(f"\n   Baseline: 0.5921")
print(f"   Change:   {(r2_score(y_test, y_pred_opt_test) - 0.5921)*100:+.2f}%")